In [8]:
import os, json, cv2, random, shutil
import numpy as np
import matplotlib.pyplot as plt
from roboflow import Roboflow
from sklearn.model_selection import train_test_split
import albumentations as A
import glob
import wandb

# Standard library and third-party imports

In [9]:
# --- CONFIGURATION ---
API_KEY = "b7edpt2HuAjVNVKXo52u"
WORKSPACE = "mohamed-uob"
PROJECT_NAME = "denim"
VERSION = 1
OUTPUT_DIR = "processed_datasets" # Added to fix the NameError

# 1. CLEANUP PREVIOUS DATA
# Find and remove folders starting with PROJECT_NAME to ensure a clean start
current_dir = os.getcwd()
for item in os.listdir(current_dir):
    if item.startswith(PROJECT_NAME) and os.path.isdir(item):
        print(f"Deleting old folder: {item}")
        shutil.rmtree(item)

# 2. FRESH DOWNLOAD
try:
    print("Connecting to Roboflow...")
    rf = Roboflow(api_key=API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
    
    print(f"Downloading dataset (COCO format, version: {VERSION})...")
    dataset = project.version(VERSION).download("coco")
    
    base_path = dataset.location
    print(f"Successful download to: {base_path}")

    # 3. SEARCH AND VERIFY ANNOTATIONS
    print("Searching for annotation files...")
    json_files = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))
                print(f"Found file: {file} in {root}")

    if not json_files:
        print("--- ERROR MESSAGE ---")
        print("Download finished, but no JSON file was found.")
        print("Please check your project annotations in the browser!")
    else:
        json_path = json_files[0]
        raw_dir = os.path.dirname(json_path)
        
        with open(json_path, 'r') as f:
            coco_data = json.load(f)
            
        print(f"\nSUCCESS! Loaded {len(coco_data['images'])} images.")
        print(f"Source directory for images: {raw_dir}")

except Exception as e:
    print(f"An error occurred during download: {e}")

Deleting old folder: denim-1
Connecting to Roboflow...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to denim-1 in coco:: 100%|██████████| 9305/9305 [00:11<00:00, 817.18it/s] 


Successful download to: d:\BME MSc\2025_26_2\Haladó adatelemzés\VITMMB10-halado-adatelemzesi-labor\denim-1
Searching for annotation files...
Found file: _annotations.coco.json in d:\BME MSc\2025_26_2\Haladó adatelemzés\VITMMB10-halado-adatelemzesi-labor\denim-1\train
Found file: _annotations.coco.json in d:\BME MSc\2025_26_2\Haladó adatelemzés\VITMMB10-halado-adatelemzesi-labor\denim-1\valid

SUCCESS! Loaded 8994 images.
Source directory for images: d:\BME MSc\2025_26_2\Haladó adatelemzés\VITMMB10-halado-adatelemzesi-labor\denim-1\train


In [10]:
# Map categories and annotations
categories = {cat['id']: cat['name'] for cat in coco_data['categories']}
caries_id = next((id for id, name in categories.items() if 'caries' in name.lower()), None)

# Keep only teeth labeled 1-32
tooth_ids = [cid for cid, name in categories.items() if name.isdigit() and 1 <= int(name) <= 32]

img_to_annots = {img['id']: [] for img in coco_data['images']}
for ann in coco_data['annotations']:
    img_to_annots[ann['image_id']].append(ann)

# Separate images by task
module1_images = [] # For tooth identification
module2_images = [] # For caries detection

for img in coco_data['images']:
    annots = img_to_annots[img['id']]
    if any(a['category_id'] in tooth_ids for a in annots):
        module1_images.append(img)
    if any(a['category_id'] == caries_id for a in annots):
        module2_images.append(img)

print(f"Module 1 (Tooth ID): {len(module1_images)} images selected.")
print(f"Module 2 (Caries): {len(module2_images)} images selected.")

Module 1 (Tooth ID): 6155 images selected.
Module 2 (Caries): 547 images selected.


In [11]:
IMG_SIZE = (640, 640)

# Augmentation pipeline based on the lab plan
aug_pipeline = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(p=0.3), 
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def apply_preprocessing(img):
    """
    Standard medical imaging preprocessing:
    Grayscale conversion -> Min-Max normalization -> Gaussian blur for noise reduction.
    """
    # 1. Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Normalization: Scale pixel values between 0-255
    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    
    # 3. Gaussian smoothing: Suppress noise while preserving edges
    denoised = cv2.GaussianBlur(norm, (3, 3), 0)
    
    # Convert back to 3 channels as required by YOLO models
    return cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)

In [12]:
def export_dataset(img_list, folder_name, target_ids, is_caries_module=False):
    print(f"Exporting: {folder_name}...")
    # 80-20% Train-Val split
    train, val = train_test_split(img_list, test_size=0.2, random_state=42)

    for split_name, subset in [('train', train), ('val', val)]:
        img_out = f"{OUTPUT_DIR}/{folder_name}/{split_name}/images"
        lbl_out = f"{OUTPUT_DIR}/{folder_name}/{split_name}/labels"
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)

        for img_info in subset:
            img_path = os.path.join(raw_dir, img_info['file_name'])
            img = cv2.imread(img_path)
            if img is None: continue

            bboxes, labels = [], []
            for ann in img_to_annots[img_info['id']]:
                if ann['category_id'] in target_ids or (is_caries_module and ann['category_id'] == caries_id):
                    x, y, w, h = ann['bbox']
                    # Clipping: Force coordinates between [0, 1]
                    xc = max(0, min(1, (x + w/2) / img_info['width']))
                    yc = max(0, min(1, (y + h/2) / img_info['height']))
                    wn = max(0, min(1, w / img_info['width']))
                    hn = max(0, min(1, h / img_info['height']))
                    
                    if wn > 0 and hn > 0:
                        bboxes.append([xc, yc, wn, hn])
                        if is_caries_module:
                            # Label 1 for caries, 0 for healthy
                            labels.append(1 if ann['category_id'] == caries_id else 0)
                        else:
                            labels.append(ann['category_id'])

            if bboxes:
                try:
                    processed = apply_preprocessing(img)
                    transformed = aug_pipeline(image=processed, bboxes=bboxes, class_labels=labels)
                    
                    fn = img_info['file_name'].split('.')[0] + ".png"
                    cv2.imwrite(f"{img_out}/{fn}", transformed['image'])
                    with open(f"{lbl_out}/{fn.replace('.png', '.txt')}", "w") as f:
                        for b, l in zip(transformed['bboxes'], transformed['class_labels']):
                            f.write(f"{int(l)} {' '.join([f'{c:.6f}' for c in b])}\n")
                except: continue

# Execution
export_dataset(module1_images, "Module1_Segmentation", tooth_ids)
export_dataset(module2_images, "Module2_Caries", tooth_ids, is_caries_module=True)
print("\nExport successful!")

Exporting: Module1_Segmentation...
Exporting: Module2_Caries...

Export successful!


In [13]:
def plot_samples(module_name):
    path = f"{OUTPUT_DIR}/{module_name}/train/images/*.png"
    sample_files = glob.glob(path)[:3]
    
    plt.figure(figsize=(15, 5))
    for i, img_p in enumerate(sample_files):
        img = cv2.imread(img_p)
        plt.subplot(1, 3, i+1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"{module_name} Sample {i+1}")
        plt.axis('off')
    plt.show()

plot_samples("Module1_Segmentation")
plot_samples("Module2_Caries")

<Figure size 1500x500 with 0 Axes>

<Figure size 1500x500 with 0 Axes>

In [ ]:
# 1. Initialize project in the cloud
run = wandb.init(project="dental-caries-detection", job_type="data-preprocessing")

# 2. Log the processed dataset as an Artifact
artifact = wandb.Artifact(name="dental-preprocessed-data", type="dataset")

# Add the entire directory created by the export function
artifact.add_dir(OUTPUT_DIR)

# 3. Start upload
run.log_artifact(artifact)

# 4. Optional: Log a sample image to the Dashboard
sample_files = glob.glob(f"{OUTPUT_DIR}/Module2_Caries/train/images/*.png")

if sample_files:
    sample_path = sample_files[0]
    sample_img = cv2.imread(sample_path)
    # Log the image to verify augmentations remotely
    wandb.log({"preprocessing_result": wandb.Image(sample_img, caption="Augmented & Preprocessed Sample")})
    print(f"Sample image uploaded: {sample_path}")
else:
    print("No images found for upload in the specified directory.")

# Finish run
run.finish()